# Exercise 0 - PydanticAI and OpenRouter


1. Product extractor

1a) Read the products.json and store all of its descriptions into a list

# Importera bibliotek

In [ ]:
import json
from pathlib import Path

# Läs filen

In [ ]:
file_path = Path("data/products.json")

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Kolla hur datan ser ut
data[:2]

# Extraherar beskrivningar

In [ ]:
descriptions = [item["description"] for item in data]

print(descriptions[:3])

1b) Loop through this list and extract structured output from each of the descriptions. The structure should have the fields: name, price, category, in_stock and description

# Skapa Pydantic modell

In [ ]:
from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    category: str
    in_stock: bool
    description: str

# Skapa agent

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model="openrouter:nvidia/nemotron-nano-12b-v2-vl:free",  # eller din modell via OpenRouter
    system_prompt="""
    Extract structured product information from the given text.

    Return:
    - name
    - price (number)
    - category
    - in_stock (true/false)
    - description (clean summary)
    """
)

async def extract_product(description_text: str) -> Product:
    result = await agent.run(
        description_text,
        output_type=Product
    )

    return result.output

Loopa genom alla beskrivningar & kolla resultat

In [ ]:
structured_products = []

for desc in descriptions:
  product = await extract_product(desc)
  structured_products.append(product)

structured_products[:1]

1c) Now try to validate programmatically if the output fields are correct.

In [ ]:
from pydantic import BaseModel, Field, field_validator

class ProductInfo(BaseModel):
    name: str = Field(min_length=1)
    price: float
    category: str = Field(min_length=1)
    in_stock: bool
    description: str = Field(min_length=5)

    @field_validator("price")
    @classmethod
    def validate_price(cls, v):
        if v < 0:
            raise ValueError("price must be non-negative")
        return v
    
    @field_validator("category")
    @classmethod
    def validate_category(cls, v):
        allowed_categories = {
            "electronics",
            "clothing",
            "home",
            "beauty",
            "sports",
            "books",
            "toys",
            "food",
            "other",
            
        }
        if v.lower() not in allowed_categories:
            raise ValueError(f"invalid category: {v}")
        return v

In [ ]:
validated_products = []
failed_products = []

for i, desc in enumerate(descriptions):
    try:
        result = await agent.run(
            desc,
            output_type=ProductInfo
        )
        validated_products.append(result.output)
    except Exception as e:
        failed_products.append({
            "index": i,
            "description": desc,
            "error": str(e)
        })
print ("Valid:", len(validated_products))
print ("Failed:", len(failed_products))